In [1]:
import os
os.chdir("/root/EdgeTRM")
print(os.getcwd())

!git config --global --add safe.directory /root/EdgeTRM
# !rm -rf /root/EdgeTRM/EdgeTRM
 


/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f


In [2]:
# !git fetch origin
# !git reset --hard origin/main
# !git pull origin main

In [3]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

repo_root: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f
trm_root: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [4]:

!uv pip install --system {trm_root}

Using Python 3.12.6 environment at: /usr/local
Resolved 64 packages in 1.99s
Building antlr4-python3-runtime==4.9.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
coolname   ------------------------------     0 B/36.96 KiB
Bui

In [5]:
import os
os.chdir('/root/EdgeTRM/TinyRecursiveModels')


In [12]:
import torch
import yaml
from trm import TinyRecursiveReasoningModel_ACTV1

def load_arc_model(checkpoint_path, config_text):
    # 1. Parse the YAML provided in your prompt
    raw_config = yaml.safe_load(config_text)
    
    # 2. Manually map and extract the required fields
    # We take the values from the 'arch' section of your YAML
    arch = raw_config['arch']
    
    # Construct the exact dictionary required by TinyRecursiveReasoningModel_ACTV1Config
    final_config = {
        # Required fields missing or named differently in your YAML:
        "batch_size": raw_config.get('global_batch_size', 512), # Map global_batch_size to batch_size
        "seq_len": 1024,                # Typical for ARC models; not found in your YAML
        "num_puzzle_identifiers": 30670, # Size of the task lookup table; not found in your YAML
        "vocab_size": 12,               # 10 colors + 1 padding/special token for ARC
        
        # Fields from your 'arch' section:
        "H_cycles": arch['H_cycles'],
        "L_cycles": arch['L_cycles'],
        "H_layers": arch['H_layers'],
        "L_layers": arch['L_layers'],
        "hidden_size": arch['hidden_size'],
        "expansion": arch['expansion'],
        "num_heads": arch['num_heads'],
        "pos_encodings": arch['pos_encodings'],
        "halt_max_steps": arch['halt_max_steps'],
        "halt_exploration_prob": arch['halt_exploration_prob'],
        
        # Optional fields from your YAML:
        "forward_dtype": arch.get('forward_dtype', 'bfloat16'),
        "mlp_t": arch.get('mlp_t', False),
        "puzzle_emb_ndim": arch.get('puzzle_emb_ndim', 512),
        "puzzle_emb_len": arch.get('puzzle_emb_len', 16),
        "no_ACT_continue": arch.get('no_ACT_continue', True)
    }

    # 1. Initialize the model
    model = TinyRecursiveReasoningModel_ACTV1(config_dict=final_config)
    
    # 2. Load the raw state_dict
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    
    # 3. If the state_dict is nested under a 'model' key (common in these checkpoints)
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    # 4. Strip the unwanted prefix added by torch.compile
    # The error showed 'unwanted' keys starting with: _orig_mod.model.
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            # Remove exactly the length of the prefix
            new_key = k[len(unwanted_prefix):]
            clean_state_dict[new_key] = v
        else:
            clean_state_dict[k] = v
            
    # 5. Load the cleaned state_dict
    model.load_state_dict(clean_state_dict)

    model.eval()
    print("Prefixes stripped and model loaded successfully!")
    return model



# Your config as a string (handling the formatting from your prompt)
config_data = """
arch:
  H_cycles: 3
  H_layers: 0
  L_cycles: 4
  L_layers: 2
  expansion: 4
  forward_dtype: bfloat16
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  num_heads: 8
  pos_encodings: rope
  puzzle_emb_len: 16
  puzzle_emb_ndim: 512
global_batch_size: 512
"""

checkpoint = "step_7471" # Ensure this file is in your directory

try:
    model = load_arc_model(checkpoint, config_data)
    print("Model successfully loaded!")
except Exception as e:
    print(f"Error: {e}")

model

Prefixes stripped and model loaded successfully!
Model successfully loaded!


TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)

In [14]:
# ── Cell 9: Sparsity Audit ────────────────────────────────────────────────────
def count_zero_params(m):
    total, zeros = 0, 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros, total

for name, m in [('Original', model)]:
    z, t = count_zero_params(m)
    print(f'{name:<15} zero={z:,}/{t:,}  ({100*z/t:.1f}%)')

Original        zero=512/6,829,058  (0.0%)


In [6]:
%%writefile eval-arc.py

from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil
import copy

import torch
import torch.distributed as dist
from torch import nn
from torch.utils.data import DataLoader

import tqdm
#import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from adam_atan2_pytorch import AdamAtan2

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed
from models.ema import EMAHelper


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str
    loss: LossConfig


class EvaluatorConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra="allow")
    name: str


class PretrainConfig(pydantic.BaseModel):
    # Config
    arch: ArchConfig
    # Data
    data_paths: List[str]
    data_paths_test: List[str] = []
    # Evaluators
    evaluators: List[EvaluatorConfig] = []

    # Hyperparams
    global_batch_size: int
    epochs: int

    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int

    weight_decay: float
    beta1: float
    beta2: float

    # Puzzle embedding
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float

    # Names
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    load_checkpoint: Optional[str] = None
    checkpoint_path: Optional[str] = None

    # Extras
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    min_eval_interval: Optional[int] = 0 # when to start eval
    eval_save_outputs: List[str] = []

    ema: bool = False # use Exponential-Moving-Average
    ema_rate: float = 0.999 # EMA-rate
    freeze_weights: bool = False # If True, freeze weights and only learn the embeddings

@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any

    step: int
    total_steps: int


def create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed,
        dataset_paths=config.data_paths_test if len(config.data_paths_test)>0 and split=="test" else config.data_paths,
        rank=rank,
        num_replicas=world_size,
        **kwargs
    ), split=split)
    dataloader = DataLoader(
        dataset,
        batch_size=None,
        num_workers=1,
        prefetch_factor=8,
        pin_memory=True,
        persistent_workers=True
    )
    return dataloader, dataset.metadata


def create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,  # type: ignore
        batch_size=config.global_batch_size // world_size,
        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,
        causal=False  # Non-autoregressive
    )

    # Instantiate model with loss head
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    with torch.device("cuda"):
        model: nn.Module = model_cls(model_cfg)
        print(model)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model)  # type: ignore

        # Load checkpoint
        if rank == 0:
            load_checkpoint(model, config)

        # Broadcast parameters from rank 0
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    # Optimizers and lr
    if config.arch.puzzle_emb_ndim == 0:
        optimizers = [
            AdamAtan2(
                model.parameters(),
                lr=0.0001,  # Needs to be set by scheduler
                weight_decay=config.weight_decay,
                betas=(config.beta1, config.beta2)
            )
        ]
        optimizer_lrs = [
            config.lr
        ]
    elif config.freeze_weights:
        optimizers = [
            CastedSparseEmbeddingSignSGD_Distributed(
                model.model.puzzle_emb.buffers(),  # type: ignore
                lr=0,  # Needs to be set by scheduler
                weight_decay=config.puzzle_emb_weight_decay,
                world_size=world_size
            )
        ]
        optimizer_lrs = [
            config.puzzle_emb_lr
        ]
    else:
        optimizers = [
            CastedSparseEmbeddingSignSGD_Distributed(
                model.model.puzzle_emb.buffers(),  # type: ignore
                lr=0,  # Needs to be set by scheduler
                weight_decay=config.puzzle_emb_weight_decay,
                world_size=world_size
            ),
            AdamAtan2(
                model.parameters(),
                lr=0.0001,  # Needs to be set by scheduler
                weight_decay=config.weight_decay,
                betas=(config.beta1, config.beta2)
            )
        ]
        optimizer_lrs = [
            config.puzzle_emb_lr,
            config.lr
        ]

    return model, optimizers, optimizer_lrs

def mix_weights_direct(device, alpha, net, nets):
    sd = []
    for i in range(len(nets)):
        sd += [nets[i].state_dict()]
    sd_alpha = {}
    for k in sd[0].keys():
        comb_net = alpha[0]*sd[0][k].to(device)
        for i in range(1,len(nets)):
            comb_net += alpha[i]*sd[i][k].to(device)
        sd_alpha[k] =  comb_net
    net.load_state_dict(sd_alpha)
    return net

def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))

    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    # Estimated total training steps
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)

    # Model
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, rank=rank, world_size=world_size)

    return TrainState(
        step=0,
        total_steps=total_steps,

        model=model,
        optimizers=optimizers,
        optimizer_lrs=optimizer_lrs,
        carry=None
    )


def save_train_state(config: PretrainConfig, train_state: TrainState):
    # FIXME: Only saved model.
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))


def load_checkpoint(model: nn.Module, config: PretrainConfig):
    if config.load_checkpoint is not None:
        print(f"Loading checkpoint {config.load_checkpoint}")

        # Load state dict
        state_dict = torch.load(config.load_checkpoint, map_location="cuda")

        # Resize and reset puzzle emb if needed
        puzzle_emb_name = "_orig_mod.model.inner.puzzle_emb.weights"
        expected_shape: torch.Size = model.model.puzzle_emb.weights.shape  # type: ignore
        if puzzle_emb_name in state_dict:
            puzzle_emb = state_dict[puzzle_emb_name]
            if puzzle_emb.shape != expected_shape:
                print(f"Resetting puzzle embedding as shape is different. Found {puzzle_emb.shape}, Expected {expected_shape}")
                # Re-initialize using mean
                state_dict[puzzle_emb_name] = (
                    torch.mean(puzzle_emb, dim=0, keepdim=True).expand(expected_shape).contiguous()
                )
        model.load_state_dict(state_dict, assign=True)


def compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step,
        base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps,
        min_ratio=config.lr_min_ratio
    )



def create_evaluators(config: PretrainConfig, eval_metadata: PuzzleDatasetMetadata) -> List[Any]:
    data_paths = config.data_paths_test if len(config.data_paths_test)>0 else config.data_paths
    # Initialize evaluators
    #print('evaluator', data_paths, config.evaluators)
    evaluators = []
    for cfg in config.evaluators:
        for data_path in data_paths:
            klass = load_model_class(cfg.name, "evaluators.")
            #print('klass', klass)
            cls = klass(
                data_path=data_path, eval_metadata=eval_metadata, **cfg.__pydantic_extra__
            )  # type: ignore
            #print('cls', cls)
            evaluators.append(cls)

    return evaluators

def train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):
    train_state.step += 1
    if train_state.step > train_state.total_steps:  # At most train_total_steps
        return

    # To device
    batch = {k: v.cuda() for k, v in batch.items()}

    # Init carry if it is None
    if train_state.carry is None:
        with torch.device("cuda"):
            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore

    # Forward
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])

    ((1 / global_batch_size) * loss).backward()

    # Allreduce
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad)
            
    # Apply optimizer
    lr_this_step = None    
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)

        for param_group in optim.param_groups:
            param_group['lr'] = lr_this_step
            
        optim.step()
        optim.zero_grad()

    # Reduce metrics
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())

        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
        # Reduce and reconstruct
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1:
            dist.reduce(metric_values, dst=0)

        if rank == 0:
            metric_values = metric_values.cpu().numpy()
            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}
            
            # Postprocess
            count = max(reduced_metrics["count"], 1)  # Avoid NaNs
            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}

            reduced_metrics["train/lr"] = lr_this_step
            return reduced_metrics

def evaluate(
    config: PretrainConfig,
    train_state: TrainState,
    eval_loader: torch.utils.data.DataLoader,
    eval_metadata: PuzzleDatasetMetadata,
    evaluators: List[Any],
    rank: int,
    world_size: int,
    cpu_group: Optional[dist.ProcessGroup],
):
    reduced_metrics = None

    with torch.inference_mode():
        return_keys = set(config.eval_save_outputs)
        for evaluator in evaluators:
            evaluator.begin_eval()
            return_keys.update(evaluator.required_outputs)

        # Run evaluation
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}

        save_preds = {}

        metric_keys = []
        metric_values = None

        carry = None
        processed_batches = 0
        
        for set_name, batch, global_batch_size in eval_loader:
            processed_batches += 1
            if rank == 0:
                print(f"Processing batch {processed_batches}: {set_name}")
            
            # To device
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)  # type: ignore

            # Forward
            inference_steps = 0
            while True:
                carry, loss, metrics, preds, all_finish = train_state.model(
                    carry=carry, batch=batch, return_keys=return_keys
                )
                inference_steps += 1

                if all_finish:
                    break

            if rank == 0:
                print(f"  Completed inference in {inference_steps} steps")

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        save_preds.setdefault(k, [])
                        save_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory

            for evaluator in evaluators:
                evaluator.update_batch(batch, preds)

            del carry, loss, preds, batch, all_finish

            # Aggregate metrics
            set_id = set_ids[set_name]

            if metric_values is None:
                metric_keys = list(
                    sorted(metrics.keys())
                )  # Sort keys to guarantee all processes use the same order.
                metric_values = torch.zeros(
                    (len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda"
                )

            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])

            del metrics

        # concatenate save preds
        save_preds = {k: torch.cat(v, dim=0) for k, v in save_preds.items()}

        # Save preds
        if config.checkpoint_path is not None and len(save_preds):
            # Each rank save predictions independently
            os.makedirs(os.path.dirname(config.checkpoint_path), exist_ok=True)
            torch.save(
                save_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}")
            )

        del save_preds

        # Reduce to rank 0
        if metric_values is not None:
            if world_size > 1:
                dist.reduce(metric_values, dst=0)

            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {
                    set_name: {
                        metric_name: reduced_metrics[set_id, metric_id]
                        for metric_id, metric_name in enumerate(metric_keys)
                    }
                    for set_id, set_name in enumerate(set_ids)
                }

                # Postprocess
                for set_name, m in reduced_metrics.items():
                    count = m.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in m.items()}

        # Run evaluators
        if rank == 0:
            print(f"\nRunning {len(evaluators)} evaluator(s)...")
            
        for i, evaluator in enumerate(evaluators):
            if rank == 0:
                print(f"Running evaluator {i+1}/{len(evaluators)}: {evaluator.__class__.__name__}")
                
            # Path for saving
            evaluator_save_path = None
            if config.checkpoint_path is not None:
                evaluator_save_path = os.path.join(
                    config.checkpoint_path,
                    f"evaluator_{evaluator.__class__.__name__}_step_{train_state.step}",
                )
                os.makedirs(evaluator_save_path, exist_ok=True)

            # Run and log
            metrics = evaluator.result(evaluator_save_path, rank=rank, world_size=world_size, group=cpu_group)
            if rank == 0 and metrics is not None:
                if reduced_metrics is None:
                    reduced_metrics = {}

                reduced_metrics.update(metrics)
                print(f"  Completed {evaluator.__class__.__name__}")
                
        if rank == 0:
            print("All evaluators completed!")

    return reduced_metrics

def save_code_and_config(config: PretrainConfig):
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)

    # Copy code
    code_list = [
        get_model_source_path(config.arch.name),
        get_model_source_path(config.arch.loss.name)
    ]
    for code_file in code_list:
        if code_file is not None:
            code_name = os.path.basename(code_file)

            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))

    # Dump config as yaml
    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")
    with open(config_file, "wt") as f:
        yaml.dump(config.model_dump(), f)

    # Log code
    print(config.checkpoint_path)


def load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)  # type: ignore

        # Naming
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_paths[0]).capitalize()}-ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)

        objects = [config]

    if world_size > 1:
        dist.broadcast_object_list(objects, src=0)

    return objects[0]  # type: ignore


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config: DictConfig):
    RANK = 0
    WORLD_SIZE = 1
    CPU_PROCESS_GROUP = None

    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
        
        # CPU GLOO process group
        CPU_PROCESS_GROUP = dist.new_group(backend="gloo")
        assert (
            dist.get_rank(CPU_PROCESS_GROUP) == RANK and dist.get_world_size(CPU_PROCESS_GROUP) == WORLD_SIZE
        )

    # Load sync'ed config
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)

    # Seed RNGs to ensure consistency
    torch.random.manual_seed(config.seed + RANK)

    # Dataset
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter

    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."

    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    try:
        eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    except:
        print("NO EVAL DATA FOUND")
        eval_loader = eval_metadata = None

    try:
        evaluators = create_evaluators(config, eval_metadata)
    except:
        print("No evaluator found")
        evaluators = []

    # Train state
    train_state = init_train_state(config, train_metadata, rank=RANK, world_size=WORLD_SIZE)

    # Progress bar and logger
    progress_bar = None
    ema_helper = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)
        print({"num_params": sum(x.numel() for x in train_state.model.parameters())})
        save_code_and_config(config)
    if config.ema:
        print('Setup EMA')
        ema_helper = EMAHelper(mu=config.ema_rate)
        ema_helper.register(train_state.model)

    # Training Loop
    for _iter_id in range(total_iters):
        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")

        ############ Train Iter
        if RANK == 0:
            print("TRAIN")
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                #print(metrics, train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore
            if config.ema:
                ema_helper.update(train_state.model)

        if _iter_id >= config.min_eval_interval:
            ############ Evaluation
            if RANK == 0:
                print("EVALUATE")
            if config.ema:
                print("SWITCH TO EMA")
                train_state_eval = copy.deepcopy(train_state)
                train_state_eval.model = ema_helper.ema_copy(train_state_eval.model)
            else:
                train_state_eval = train_state
            train_state_eval.model.eval()
            metrics = evaluate(config, 
                train_state_eval, 
                eval_loader, 
                eval_metadata, 
                evaluators,
                rank=RANK, 
                world_size=WORLD_SIZE,
                cpu_group=CPU_PROCESS_GROUP)

            if RANK == 0 and metrics is not None:
                print(metrics, train_state.step)
                
            ############ Checkpointing
            if RANK == 0:
                print("SAVE CHECKPOINT")
            if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
                save_train_state(config, train_state_eval)

            if config.ema:
                del train_state_eval

    # finalize
    if dist.is_initialized():
        dist.destroy_process_group()



if __name__ == "__main__":
    launch()

Overwriting eval-arc.py


In [7]:
! mkdir data1
! cp /root/EdgeTRM/arc-prize-2025/arc-agi_test_challenges.json data1

mkdir: cannot create directory ‘data1’: File exists


In [8]:
# !ls /root/EdgeTRM/arc-prize-2024
!ls /root/EdgeTRM/arc-prize-2025

arc-agi_evaluation_challenges.json  arc-agi_training_solutions.json
arc-agi_evaluation_solutions.json   arc_2025_setup
arc-agi_test_challenges.json	    sample_submission.json
arc-agi_training_challenges.json


In [9]:
import os
import json

# # 1. Path to the 2024 competition data
# SOURCE_DIR = '/root/EdgeTRM/arc-prize-2024'
# WORKING_DIR = '/root/EdgeTRM/arc-prize-2024/arc_2024_setup'

# # # 1. Path to the 2025 competition data
SOURCE_DIR = '/root/EdgeTRM/arc-prize-2025'
WORKING_DIR = '/root/EdgeTRM/arc-prize-2025/arc_2025_setup'
os.makedirs(WORKING_DIR, exist_ok=True)

def setup_arc_2024(subset_name, has_solutions=True):
    challenge_src = os.path.join(SOURCE_DIR, f'arc-agi_{subset_name}_challenges.json')
    solution_src = os.path.join(SOURCE_DIR, f'arc-agi_{subset_name}_solutions.json')
    
    challenge_link = os.path.join(WORKING_DIR, f'arc_{subset_name}_challenges.json')
    solution_link = os.path.join(WORKING_DIR, f'arc_{subset_name}_solutions.json')

    # Link Challenges
    if os.path.lexists(challenge_link): os.remove(challenge_link)
    if os.path.exists(challenge_src):
        os.symlink(challenge_src, challenge_link)
    else:
        print(f"File not found: {challenge_src}")
        return

    # Handle Solutions
    if has_solutions and os.path.exists(solution_src):
        if os.path.lexists(solution_link): os.remove(solution_link)
        os.symlink(solution_src, solution_link)
        print(f"Linked real data for: {subset_name}")
    else:
        # Create dummy solutions as simple lists of grids for the builder
        with open(challenge_src, 'r') as f:
            challenges = json.load(f)
        dummy_data = {k: [[[0]] for _ in v['test']] for k, v in challenges.items()}
        with open(solution_link, 'w') as f:
            json.dump(dummy_data, f)
        print(f"Linked challenges and created dummy solutions for: {subset_name}")

# Training and Evaluation have real solutions in the 2024 folder
setup_arc_2024('training', has_solutions=True)
setup_arc_2024('evaluation', has_solutions=True)

# Test does not have a solutions file in the competition folder
setup_arc_2024('test', has_solutions=False)

print(f"\nUse this prefix for your build script: {WORKING_DIR}/arc")

Linked real data for: training
Linked real data for: evaluation
Linked challenges and created dummy solutions for: test

Use this prefix for your build script: /root/EdgeTRM/arc-prize-2025/arc_2025_setup/arc


In [10]:
! python -m dataset.build_arc_dataset \
  --input-file-prefix  /root/EdgeTRM/arc-prize-2025/arc_2025_setup/arc \
  --output-dir data1/arc2test-aug-128 \
  --subsets test \
  --test-set-name test \
  --num-aug 128

# !python -m dataset.build_arc_dataset \
#   --input-file-prefix  /kaggle/working/EdgeTRM/arc-prize-2025/arc_2025_setup/arc \
#   --output-dir /root/EdgeTRM/data/arc_processed \
#   --subsets training evaluation test \
#   --test-set-name evaluation \
#   --num-aug 8

[Puzzle 332efdb3] augmentation not full, only 9
[Puzzle 2697da3f] augmentation not full, only 72
[Puzzle 28e73c20] augmentation not full, only 72
[Puzzle 1990f7a8] augmentation not full, only 72
Total puzzles: 240
Total puzzle IDs (including <blank>): 30670


In [11]:
!pwd

/__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [12]:
import os
if 1 or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !HYDRA_FULL_ERROR=1 WANDB_MODE=disabled torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
    eval-arc.py \
    arch=trm \
    data_paths="[./data1/arc2test-aug-128]" \
    arch.L_layers=2 \
    arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
    freeze_weights=False \
    +load_checkpoint=/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907 \
    +checkpoint_path=./eval_checkpoint \
    eval_interval=5000 \
    epochs=5000 \
    global_batch_size=512 \
    ema=True \
    lr_warmup_steps=200 \
    lr=0.0001
else:
    !HYDRA_FULL_ERROR=1 WANDB_MODE=disabled torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
    eval-arc.py \
    arch=trm \
    data_paths="[./data1/arc2test-aug-128]" \
    arch.L_layers=2 \
    arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
    freeze_weights=False \
    +load_checkpoint=/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907 \
    +checkpoint_path=./eval_checkpoint \
    eval_interval=100 \
    epochs=100 \
    global_batch_size=512 \
    ema=True \
    lr_warmup_steps=200 \
    lr=0.0001   

TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)
Loading checkpoint /root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907
  0%|                                                  | 0/7481 [00:00<?, ?it/s]{'num_params': 6829058}
./eval_checkpoint
Setup EMA
[Rank 0, World Size 1]: Epoch 0
TRAIN
100%|████████████████████████████████████▉| 7471/7481 [1:38:52<0

In [3]:
! ls eval_checkpoint*/evaluator*/submission.json 

eval_checkpoint/evaluator_ARC_step_14907/submission.json
eval_checkpoint/evaluator_ARC_step_7471/submission.json


In [8]:
!cp -f eval_checkpoint/evaluator_ARC_step_7471/submission.json ./submission1.json
!cp -f eval_checkpoint/step_7471 .
!cp -f eval_checkpoint/step_14907 .


In [15]:
ls -lh /root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907

-rw-r--r-- 1 root root 86M Apr 26 19:34 /root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907


In [8]:
import os
import json
import numpy as np

# Set to True to calculate local scores; False for a "clean" submission run
sub_eval = True 

Grid = list[list[int]]

def validate_grid(grid: Grid) -> bool:
    """Ensures grids meet competition constraints (1x1 to 30x30, colors 0-9)."""
    try:
        array = np.array(grid, dtype=np.int8)
        if array.ndim != 2:
            return False
        if not (1 <= array.shape[0] <= 30 and 1 <= array.shape[1] <= 30):
            return False
        if not np.all(np.isin(array, np.arange(10))):
            return False
        return True
    except:
        return False

# 1. Load your model's predictions
submission_file = "./submission1.json"
if not os.path.exists(submission_file):
    print(f"Error: {submission_file} not found.")
    exit()

with open(submission_file, "r") as f:
    submission = json.load(f)

# 2. Detailed Scoring Logic
if sub_eval:
    print("--- Starting Detailed Evaluation Breakdown ---")
    
    # Path Configuration
    base_path = "/root/EdgeTRM/arc-prize-2025"
    train_sol_path = os.path.join(base_path, "arc-agi_training_solutions.json")
    eval_sol_path = os.path.join(base_path, "arc-agi_evaluation_solutions.json")
    
    # Load Solution Sets Separately
    train_solutions = {}
    eval_solutions = {}
    
    if os.path.exists(train_sol_path):
        with open(train_sol_path, "r") as f:
            train_solutions = json.load(f)
        print(f"Loaded {len(train_solutions)} training solutions.")
        
    if os.path.exists(eval_sol_path):
        with open(eval_sol_path, "r") as f:
            eval_solutions = json.load(f)
        print(f"Loaded {len(eval_solutions)} evaluation solutions.")

    # Tracking lists
    train_scores = []
    eval_scores = []
    missing_tasks = []

    for puzzle_id, test_attempts in submission.items():
        # Determine which set the puzzle belongs to
        target_sol_set = None
        is_train = False
        
        if puzzle_id in train_solutions:
            target_sol_set = train_solutions
            is_train = True
        elif puzzle_id in eval_solutions:
            target_sol_set = eval_solutions
            is_train = False
        else:
            missing_tasks.append(puzzle_id)
            continue
            
        puzzle_solutions = target_sol_set[puzzle_id]
        puzzle_score = 0
        
        # Iterate through each test instance in the puzzle
        for tid, attempt_dict in enumerate(test_attempts):
            # Validate and Clean
            for key in ["attempt_1", "attempt_2"]:
                if not validate_grid(attempt_dict.get(key, [])):
                    attempt_dict[key] = [[0]]
            
            # Score (Match attempt_1 or attempt_2 to solution)
            correct_sol = puzzle_solutions[tid]
            if attempt_dict["attempt_1"] == correct_sol or attempt_dict["attempt_2"] == correct_sol:
                puzzle_score += 1
                status = "TRAIN" if is_train else "EVAL"
                print(f"✓ [{status}] Puzzle {puzzle_id} (test {tid}) is CORRECT!")

        # Append to the correct tracking list
        if is_train:
            train_scores.append(puzzle_score / len(test_attempts))
        else:
            eval_scores.append(puzzle_score / len(test_attempts))

    # Calculate Totals
    def calc_stats(score_list):
        if not score_list:
            return 0.0, 0.0
        total_correct = sum(score_list)
        accuracy = (total_correct / len(score_list)) * 100
        return total_correct, accuracy

    train_total, train_acc = calc_stats(train_scores)
    eval_total, eval_acc = calc_stats(eval_scores)
    
    combined_scores = train_scores + eval_scores
    combined_total, combined_acc = calc_stats(combined_scores)

    # 3. Final Detailed Summary Output
    print("\n" + "="*40)
    print("FINAL EVALUATION RESULTS")
    print("="*40)
    print(f"Total tasks in submission: {len(submission)}")
    
    print("\n[TRAINING SET]")
    print(f"Tasks matched:  {len(train_scores)}")
    print(f"Total Correct:  {train_total:.2f}")
    print(f"Accuracy:       {train_acc:.2f}%")
    
    print("\n[EVALUATION SET]")
    print(f"Tasks matched:  {len(eval_scores)}")
    print(f"Total Correct:  {eval_total:.2f}")
    print(f"Accuracy:       {eval_acc:.2f}%")
    
    print("\n[COMBINED TOTAL]")
    print(f"Total matched:  {len(combined_scores)}")
    print(f"Total Correct:  {combined_total:.2f}")
    print(f"Global Accuracy: {combined_acc:.2f}%")
    
    if missing_tasks:
        print(f"\n[!] Missing solution data for {len(missing_tasks)} tasks.")
    print("="*40)

# Save the cleaned submission file
with open("submission.json", "w") as f:
    json.dump(submission, f)
    print("\n'submission.json' saved and validated.")

--- Starting Detailed Evaluation Breakdown ---
Loaded 1000 training solutions.
Loaded 120 evaluation solutions.
✓ [TRAIN] Puzzle 0b148d64 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 025d127b (test 0) is CORRECT!
✓ [TRAIN] Puzzle 1e0a9b12 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 212895b5 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 1c786137 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 0520fde7 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 27a28665 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 27a28665 (test 1) is CORRECT!
✓ [TRAIN] Puzzle 27a28665 (test 2) is CORRECT!
✓ [TRAIN] Puzzle 3979b1a8 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 2f0c5170 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 22208ba4 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 4093f84a (test 0) is CORRECT!
✓ [TRAIN] Puzzle 332efdb3 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 06df4c85 (test 0) is CORRECT!
✓ [TRAIN] Puzzle 23b5c85d (test 0) is CORRECT!
✓ [TRAIN] Puzzle 0962bcdd (test 0) is CORRECT!
✓ [TRAIN] Puzzle 41e4d17e (test 0) is CORRECT!
✓ [TRAIN] Puzzle 0ca9ddb6 (test 0) is CORR

In [16]:
!WANDB_MODE=disabled torchrun --nproc_per_node=1 eval-arc.py \
  arch=trm \
  +arch.puzzle_id_max=1041208 \
  data_paths="[./data1/arc2test-aug-128]" \
  +load_checkpoint='/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907' \
  +eval_only=True \
  epochs=0 \
  evaluators="[{name: 'arc@ARC'}]" \
  +eval_save_outputs="['json']"

TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)
Loading checkpoint /root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/step_14907
0it [00:00, ?it/s]{'num_params': 6829058}
checkpoints/Arc2test-aug-128-ACT-torch/TinyRecursiveReasoningModel_ACTV1 interesting-lori
0it [00:00, ?it/s]


In [17]:

# # Example using huggingface-cli to download the specific folder
# from huggingface_hub import snapshot_download

# # This downloads only the 'arc_v1_public' folder into your local directory
# snapshot_download(
#     repo_id="arcprize/trm_arc_prize_verification",
#     allow_patterns="arc_v2_public/*",
#     local_dir="./"
# )

# print("Files downloaded to ./arc_v2_public")
# !cp /root/EdgeTRM/TinyRecursiveModels/arc_v2_public/all_config.yaml ./all_config_v2.yaml
# !cp /root/EdgeTRM/TinyRecursiveModels/arc_v2_public/step_723914 ./step_723914

# # Example using huggingface-cli to download the specific folder
# from huggingface_hub import snapshot_download

# # This downloads only the 'arc_v1_public' folder into your local directory
# snapshot_download(
#     repo_id="arcprize/trm_arc_prize_verification",
#     allow_patterns="arc_v1_public/*",
#     local_dir="./"
# )

# print("Files downloaded to ./arc_v1_public")
# !cp /root/EdgeTRM/TinyRecursiveModels/arc_v1_public/all_config.yaml ./all_config.yaml
# !cp /root/EdgeTRM/TinyRecursiveModels/arc_v1_public/step_518071 ./step_518071
# # The following code will only execute
# # successfully when compression is complete

# import kagglehub
# import os
# os.environ["KAGGLE_USERNAME"] = "pearsejim01"
# os.environ["KAGGLE_KEY"] = "KGAT_38816be21233b9082242e3cc0f0b220e"


# # Download latest version
# path = kagglehub.dataset_download("pearsejim01/trm-arc-agi-chkpt")

# print("Path to dataset files:", path)

# # Copy the folder into your current directory
# !cp -r /root/.cache/kagglehub/datasets/pearsejim01/trm-arc-agi-chkpt/versions/2/EdgeTRM .

# # # Or move it (removes the original)
# # mv /root/.cache/kagglehub/datasets/pearsejim01/trm-arc-agi-chkpt/versions/2/EdgeTRM .
# !modal volume create EdgeTRM
# import modal

# # Define the volume
# my_volume = modal.Volume.from_name("EdgeTRM", create_if_missing=True)
# !mkdir EdgeTRM/arc-prize-2024
# !cd EdgeTRM
# !modal volume put EdgeTRM ./EdgeTRM/. /
# %uv pip install kagglehub
# import kagglehub
# import os
# os.environ["KAGGLE_USERNAME"] = "pearsejim01"
# os.environ["KAGGLE_KEY"] = "KGAT_38816be21233b9082242e3cc0f0b220e"
# # Download latest version
# path = kagglehub.dataset_download("cpmpml/arc-prize-trm-031")

# print("Path to dataset files:", path)
# !cp -r  /root/.cache/kagglehub/datasets/cpmpml/arc-prize-trm-031/versions/1/step_275886 .


In [18]:
!